# Parte 7-lite — Estilometría y clasificación de hablante

**Proyecto:** Observatorio del discurso presidencial en español
**Unidades del temario:** 5.2 (representación y frecuencias), 3.6 / 5.3 (algoritmos clásicos de
aprendizaje supervisado).
**Restricciones respetadas:** cero deep learning, cero transformers, cero embeddings neuronales.
Sólo rasgos contables, TF-IDF y clasificadores lineales.

**Qué añade sobre el módulo 6.** El 6 describe temas latentes de forma no supervisada. Este módulo
aporta: (1) una métrica **predictiva** con datos retenidos (macro-F1 out-of-fold), (2) la separación
explícita **forma / contenido / periodo** mediante ablación —que convierte el confundido país = persona
en un número— y (3) un **perfil estilométrico** por líder interpretable sin entrenar nada.

**Advertencia de alcance.** Mide *cómo se escribe/habla*, no *cómo es la persona*. No se infiere
personalidad, ideología ni emociones: un discurso (y menos una transcripción) no permite eso.

| Pieza | Dónde |
|---|---|
| Paquete | `est_scripts/` |
| Pipeline | `python est_scripts/build_models.py` (~2 min con caché) |
| Artefactos y tablas | `est_scripts/artifacts/` |
| Documento | `est_scripts/est_explained.md` |

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import est_scripts.config as config
from est_scripts import classify as C
from est_scripts import evaluate as E
from est_scripts import features as F
from est_scripts.style import document_features
from ri_scripts.corpus import iter_corpus, read_doc
from ri_scripts.index import load_index

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

ix = load_index()
dataset = F.build_dataset()
y, groups = dataset.speakers, dataset.groups

print(f"documentos: {dataset.n_docs} (descartados <{config.MIN_WORDS} palabras: {ix.n_docs - dataset.n_docs})")
print(f"clases: {dict(pd.Series(y).value_counts())}")
print(f"grupos de casi-duplicados: {dataset.n_duplicate_groups} (mayor: {dataset.largest_group})")
print(f"accuracy de la clase mayoritaria: {E.baseline_majority(y):.3f}")

## 1. Rasgos estilométricos: ejemplos sobre tres discursos

TTR, MTLD y Yule's K miden **riqueza léxica**; la longitud y la puntuación miden **ritmo**; los
pronombres y marcadores miden **posición del hablante**. Todo se calcula sobre el texto crudo y se
normaliza por 1.000 palabras cuando es una frecuencia.

$$\text{TTR} = \frac{V}{N}, \qquad
K_{\text{Yule}} = 10^4\frac{\sum_i i^2 V_i - N}{N^2}, \qquad
\text{FH} = 206.84 - 0.60P - 1.02F$$

In [ ]:
filas = []
for doc_id in (1065, 2381, 2493, 2877, 4432):
    doc = read_doc(doc_id, ix.corpus_path, ix.offsets)
    feats = document_features(doc["text"])
    filas.append({
        "doc_id": doc_id,
        "hablante": doc["speaker"].split()[0],
        "fecha": doc["date"],
        "palabras": int(feats["n_words"]),
        "palabras/oración": round(feats["words_per_sent"], 1),
        "TTR": round(feats["ttr"], 3),
        "MTLD": round(feats["mtld"], 1),
        "Yule's K": round(feats["yules_k"], 1),
        "yo/1000": round(feats["first_sing_1000"], 2),
        "nosotros/1000": round(feats["first_plur_1000"], 2),
        "comas/1000": round(feats["comas"], 1),
        "Fernández-Huerta": round(feats["fernandez_huerta"], 1),
    })
display(pd.DataFrame(filas))

## 2. Perfil estilométrico por líder

Medianas de los rasgos por hablante. Las medianas (no medias) evitan que una mañanera de 20.000 palabras
arrastre la fila. Esto se puede leer sin ningún modelo: es el "retrato" descriptivo del corpus.

In [ ]:
grouped = dataset.style.groupby("speaker")
filas = []
for feature, nombre in E.PROFILE_FEATURES:
    fila = {"rasgo": nombre}
    for speaker, group in grouped:
        fila[speaker.split()[0]] = round(float(group[feature].median()), 3)
    filas.append(fila)
display(pd.DataFrame(filas))

In [ ]:
print("Rasgos más distintivos de cada líder (z de la media respecto al corpus)")
rows = []
features = [f for f, _ in E.PROFILE_FEATURES]
names = dict(E.PROFILE_FEATURES)
mean, std = dataset.style[features].mean(), dataset.style[features].std().replace(0, 1)
for speaker, group in grouped:
    z = (group[features].mean() - mean) / std
    rows.append({
        "hablante": speaker.split()[0],
        "más alto": ", ".join(f"{names[f]} ({z[f]:+.1f}σ)" for f in z.sort_values(ascending=False).head(4).index),
        "más bajo": ", ".join(f"{names[f]} ({z[f]:+.1f}σ)" for f in z.sort_values().head(4).index),
    })
display(pd.DataFrame(rows))

## 3. Validación cruzada agrupada

Dos decisiones que separan este módulo de "entrenar y mirar el accuracy":

1. **Grupos de casi-duplicados.** Si el mismo discurso está en train y en test, el F1 mide memoria.
   Se agrupa por coseno TF-IDF ≥ 0.80 y por hash del texto; `StratifiedGroupKFold` mantiene cada grupo
   en un fold.
2. **Macro-F1.** AMLO es el 50.7 % del corpus; Maduro, 30 documentos. La accuracy premia ignorar a
   Maduro; el macro-F1 no.

In [ ]:
print(f"clase mayoritaria: {E.baseline_majority(y):.3f}  (accuracy de un modelo trivial)")
print(f"grupos: {dataset.n_duplicate_groups} para {dataset.n_docs} documentos")
# tamaño de los grupos de duplicados
import collections
sizes = collections.Counter(dataset.groups.tolist())
print("tamaños de grupo:", dict(sorted(collections.Counter(sizes.values()).items())))

## 4. Ablación: ¿qué aprende el clasificador?

El mismo conjunto de modelos se entrena con bloques que miden cosas distintas. Si el mejor F1 viene de
`tema` o de `caracteres`, el modelo está detectando contenido, no forma.

| bloque | qué mide |
|---|---|
| `estilo` | riqueza, oración, puntuación, deixis |
| `funcion` | palabras de función |
| `estilo_controlado` | estilo + función (cabeza de serie de "forma") |
| `caracteres` | n-gramas de carácter (diagnóstico de fuga) |
| `caracteres_enmascarados` | ídem con nombres/gentilicios sustituidos por `X` |
| `tema` | TF-IDF de tema (contenido) |
| `fecha` | año/mes/trimestre (periodo) |
| `todo` | forma + contenido + periodo (cota superior) |

In [ ]:
abl = pd.DataFrame(json.load(open(config.EVAL_JSON, encoding="utf-8"))["ablation"])
abl["tipo"] = abl["block"].map(E.BLOCK_KIND)
display(abl[["block", "tipo", "model", "macro_f1", "fold_std", "accuracy", "seconds"]]
        .sort_values("macro_f1", ascending=False).round(3).reset_index(drop=True))

In [ ]:
# Reproducir en vivo una de las filas (SVM sobre el bloque controlado, ~7 s)
t0 = __import__("time").time()
fila = C.cv_run(dataset.blocks["estilo_controlado"], y, groups, "svm", block="estilo_controlado")
print(f"estilo_controlado / SVM: macro-F1 = {fila['macro_f1']:.3f} ± {fila['fold_f1_std']:.3f} "
      f"({__import__('time').time() - t0:.1f} s)")

## 5. El mejor modelo controlado: por clase y matriz de confusión

Se reporta por clase porque el corpus está desbalanceado: Maduro (30 documentos) pesa lo mismo que
AMLO (2,293) en el macro-F1, y su intervalo es mucho más ancho.

In [ ]:
print("Por clase")
display(pd.DataFrame([{
    "clase": r["label"].split()[0], "precisión": round(r["precision"], 3),
    "recall": round(r["recall"], 3), "F1": round(r["f1"], 3), "docs": r["support"],
} for r in fila["per_class"]]).reset_index(drop=True))

print("\nMatriz de confusión (real \\ pred)")
display(pd.DataFrame(fila["confusion"],
                     index=[l.split()[0] for l in fila["labels"]],
                     columns=[l.split()[0] for l in fila["labels"]]))

In [ ]:
# ¿Dónde están los errores? Discursos mal clasificados con su título
oof = pd.DataFrame({"doc_id": dataset.doc_ids, "real": y, "pred": fila["oof"]})
errores = oof[oof.real != oof.pred].head(8)
for row in errores.itertuples():
    print(f"{row.doc_id:5d} {row.real.split()[0]:9s} -> {row.pred.split()[0]:9s} | "
          f"{(ix.titles[row.doc_id] or '')[:80]}")

## 6. Curva de aprendizaje

¿Cuánta firma estilística hay? Con el 10 % de los discursos (~360) ya se alcanza ~0.80 de macro-F1 y
la curva se aplana cerca de 0.94: la señal es fuerte y redundante, y el techo está en la definición de
"estilo", no en la cantidad de datos.

In [ ]:
learning = pd.DataFrame(json.load(open(config.EVAL_JSON, encoding="utf-8"))["learning"])
learning["fracción"] = learning["fraction"].map(lambda x: f"{x:.0%}")
display(learning[["fracción", "macro_f1", "f1_std"]].round(3).reset_index(drop=True))

plt.figure(figsize=(6, 3.2))
plt.errorbar(learning["fracción"], learning["macro_f1"], yerr=learning["f1_std"], marker="o")
plt.ylim(0.7, 1.0)
plt.xlabel("fracción de entrenamiento")
plt.ylabel("macro-F1")
plt.title("Curva de aprendizaje — estilo controlado")
plt.tight_layout()
plt.show()

## 7. Control de ventana 2023-2024

En la ventana limpia (AMLO, Sánchez y Milei; 917 documentos) el periodo deja de ser un atajo. Si el
bloque `tema` sigue alto, el clasificador usa **de qué habla** cada líder; si el bloque de forma sube,
la distintividad está pegada al **género** (mañanera, mitin, comparecencia).

In [ ]:
window = pd.DataFrame(json.load(open(config.EVAL_JSON, encoding="utf-8"))["window"])
window["tipo"] = window["block"].map(E.BLOCK_KIND)
display(window[["block", "tipo", "macro_f1", "fold_std", "n_docs"]].round(3).reset_index(drop=True))

## 8. Diagnóstico de fuga: n-gramas de carácter

Este es el experimento que evita el autoengaño. Los n-gramas de carácter llegan a ~0.997 incluso con
los nombres enmascarados. ¿Qué están aprendiendo? Se mira el peso de los n-gramas en el SVM: si
aparecen consignas y lugares, no es estilo.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

ids = set(int(d) for d in dataset.doc_ids)
texts = [r["text"] for r in iter_corpus() if int(r["doc_id"]) in ids]
masked = [F.mask_identity(text[:config.CHAR_MAX_CHARS]) for text in texts]
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=config.CHAR_NGRAMS,
                      min_df=config.CHAR_MIN_DF, max_features=config.CHAR_MAX_FEATURES,
                      sublinear_tf=True, lowercase=True, strip_accents="unicode")
X = vec.fit_transform(masked)
vocab = np.array(vec.get_feature_names_out())
svm = LinearSVC(C=1.0, class_weight="balanced", max_iter=20000, dual=True, random_state=42)
svm.fit(X, y)
for i, clase in enumerate(svm.classes_):
    top = vocab[np.argsort(-svm.coef_[i])[:12]]
    print(f"{clase.split()[0]:10s}: {' | '.join(top)}")

Lo que se ve: apellidos sueltos (*López*, *Andrés*), derivados (*mexiquense*), consignas
(*libertad*, *patriota*), topónimos (*euro*, *Ucrania*) y fórmulas de apertura. El enmascarado cubre
nombres completos y gentilicios, pero no todas las formas derivadas ni las consignas: la fuga persiste.
Ninguno de esos n-gramas es una medida de estilo; por eso el resultado de `caracteres` se reporta como
**diagnóstico de fuga**, no como evidencia de firma estilística.

## 9. Limitaciones

1. **No mide personalidad.** Mide forma de hablar en un contexto; atribuir rasgos psicológicos a partir
   de transcripciones no es defendible.
2. **Género y fuente contaminan la forma.** Mañaneras orales de 8.000 palabras vs comparecencias
   editadas de 2.000. La ventana 2023-2024 lo hace evidente: el F1 sube al restringir a tres registros
   extremos.
3. **El tema es el atajo dominante** (0.988): es el confundido país = persona, ahora cuantificado.
4. **Casi-duplicados parciales**: el agrupamiento por coseno ≥ 0.80 no detecta pasajes compartidos
   dentro de documentos largos; MinHash+LSH por shingles (módulo 9) lo haría mejor.
5. **Maduro (30 documentos)**: F1 con intervalo amplio; se reporta con nota.
6. **Sin gold de tipo de discurso**: etiquetar género exigiría cientos de documentos y la etiqueta se
   filtraría desde el título. Fuera de alcance de la versión lite.

## 10. Reproducir

```bash
source ../env/bin/activate
python est_scripts/build_models.py            # ~2 min con features cacheadas
python est_scripts/build_models.py --force    # recalcula features (~5 min)
```

In [ ]:
for path in sorted(config.ARTIFACTS.iterdir()):
    if path.is_file():
        print(f"{path.name:24s} {path.stat().st_size / 1e6:8.2f} MB")